# Exploratory synchronized sample clustering

Cluster synchronized samples within each channel and session. This requires a documented acquisition timezone and clock synchronization confirmed by acquisition records. Neither is available here, so clustering is skipped. Set `MMS_CLOCKS_SYNCHRONIZED=1` only after synchronization has been documented. Eye readings match backward within 100 ms. Features use valid pupil readings, confidence-filtered heart rate, and intervals of 300–2000 ms.

K-means uses standardized features. PCA provides a two-dimensional display. Clusters group samples; they have no validated participant or clinical interpretation.

Use `MMS_DATA_ROOT` for another dataset and `MMS_OUTPUT_ROOT` for exports outside source data.


In [ ]:
import matplotlib.pyplot as plt
from mms import exploration as ex

SOURCE = "case-study"

for session in (1, 2, 3):
    frame = ex.synchronized_features(session, source=SOURCE)
    if frame.empty:
        continue
    for channel, samples in frame.groupby("Channel"):
        features = samples[["ibi", "heart_rate", "pupil"]].dropna()
        max_k = min(10, len(features) - 1, len(features.drop_duplicates()) - 1)
        if max_k < 2:
            print("Clustering skipped: insufficient distinct complete rows.")
            continue
        scores = []
        for k in range(2, max_k + 1):
            _, score = ex.cluster_features(features, n_clusters=k)
            scores.append((k, score))
        best_k = max(scores, key=lambda pair: pair[1])[0]
        projection, score = ex.cluster_features(features, n_clusters=best_k)
        print(
            f"Session {session}, channel {channel}: "
            f"in-sample k={best_k}, silhouette={score:.3f}"
        )
        plt.scatter(projection["PC1"], projection["PC2"], c=projection["Cluster"], s=8)
        plt.title("Exploratory sample clusters; PCA display only")
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.show()
        plt.close()
